<a href="https://colab.research.google.com/github/avnee2607/AI-project/blob/main/malaria_cnn_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Malaria cell classification with a CNN
Biocon use case: classify red blood cell images as healthy (Uninfected) or malaria-infected (Parasitized).

**Before running:** Runtime > Change runtime type > T4 GPU. Then run cells top to bottom.

## 1. Setup

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

2.21.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Download dataset (NIH Malaria Cell Images from Kaggle)

In [ ]:
!pip install -q kagglehub
import kagglehub
path = kagglehub.dataset_download('iarunava/cell-images-for-detecting-malaria')
print(path)
print(os.listdir(path))

Using Colab cache for faster access to the 'cell-images-for-detecting-malaria' dataset.
/kaggle/input/cell-images-for-detecting-malaria
['cell_images']


In [ ]:
# The folder with 'Parasitized' and 'Uninfected' may be nested. Adjust if needed.
data_dir = os.path.join(path, 'cell_images')
print(os.listdir(data_dir))

['Uninfected', 'Parasitized', 'cell_images']


## 3. Load, split and view the data

In [ ]:
IMG = (64, 64)
BATCH = 32

train_ds = keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset='training', seed=42,
    image_size=IMG, batch_size=BATCH, label_mode='binary')
val_ds = keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset='validation', seed=42,
    image_size=IMG, batch_size=BATCH, label_mode='binary')

class_names = train_ds.class_names
print(class_names)  # ['Parasitized', 'Uninfected']

plt.figure(figsize=(8, 8))
for images, labels in train_ds.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[int(labels[i])])
        plt.axis('off')
plt.show()

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

Found 55116 files belonging to 3 classes.


ValueError: When passing `label_mode="binary"`, there must be exactly 2 class_names. Received: class_names=['Parasitized', 'Uninfected', 'cell_images']

## 4. Build the CNN

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(64, 64, 3)),
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.1),
    layers.Rescaling(1./255),

    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])
model.summary()

## 5. Train

In [ ]:
early = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(train_ds, validation_data=val_ds,
                    epochs=15, callbacks=[early])

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.title('Accuracy'); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title('Loss'); plt.legend()
plt.show()

## 6. Evaluate (confusion matrix, precision, recall)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

y_true, y_pred = [], []
for x, y in val_ds:
    p = model.predict(x, verbose=0)
    y_pred.extend((p > 0.5).astype(int).flatten())
    y_true.extend(y.numpy().astype(int).flatten())

print(classification_report(y_true, y_pred, target_names=class_names))
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot()
plt.show()

## 7. Predict on a single image and save the model

In [ ]:
model.save('malaria_cnn.keras')

folder = os.path.join(data_dir, 'Parasitized')
sample = os.path.join(folder, os.listdir(folder)[0])
img = keras.utils.load_img(sample, target_size=IMG)
arr = np.expand_dims(keras.utils.img_to_array(img), 0)

prob = float(model.predict(arr)[0][0])
# class 0 = Parasitized, class 1 = Uninfected
label = 'Uninfected' if prob > 0.5 else 'Parasitized'
print(label, 'confidence:', round(max(prob, 1 - prob), 3))
plt.imshow(img); plt.axis('off'); plt.show()

## 8. (Optional) Gradio demo

In [ ]:
!pip install -q gradio
import gradio as gr

def predict(im):
    a = tf.image.resize(im, IMG)[None]
    p = float(model.predict(a)[0][0])
    return {'Uninfected': p, 'Parasitized': 1 - p}

gr.Interface(predict, gr.Image(), gr.Label(), title='Malaria cell classifier').launch()